# 03 - Train Africa Level 2 Persistence and Region-Only Controls

In [ ]:
from pathlib import Path
import sys
# Resolve local package imports from common notebook launch locations.
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT / 'src'))

import pandas as pd

from grace_gnn.config import EXPERIMENT_REGION, LAGGED_DATASET_CSV, REGION_CORRELATION_MATRIX_CSV, REGION_CORRELATION_PAIRS_CSV, REGION_PREDICTIONS_CSV, TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION, RANDOM_SEED, ensure_dirs
from grace_gnn.correlation import region_correlation_matrix, region_correlation_pairs
from grace_gnn.features import feature_columns, filter_region
from grace_gnn.splits import chronological_fraction_split
from grace_gnn.models import predict_persistence, train_correlation_neighbor, train_mlp, train_random_forest, train_ridge, train_xgboost, torch_available
from grace_gnn.evaluate import prediction_frame

ensure_dirs()
# Load the regional lagged dataset used by all baseline models.
if not LAGGED_DATASET_CSV.exists():
    print(f'Missing {LAGGED_DATASET_CSV}. Run notebook 02 first.')
    lagged = None
else:
    lagged = pd.read_csv(LAGGED_DATASET_CSV, parse_dates=['date'])
    lagged = filter_region(lagged, EXPERIMENT_REGION)
    print(f'Loaded {len(lagged):,} Africa Level 2 samples across {lagged["basin_id"].nunique()} regions')

In [ ]:
if lagged is not None:
    # Keep validation and test periods chronological to avoid temporal leakage.
    splits = chronological_fraction_split(lagged, TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION)
    total = sum(len(frame) for frame in splits.values())
    for name, frame in splits.items():
        pct = len(frame) / total if total else 0
        print(name, len(frame), f'{pct:.1%}', frame['date'].min() if len(frame) else None, frame['date'].max() if len(frame) else None)
    features = feature_columns(lagged)
    prediction_parts = []
    # Start with persistence as the no-learning reference forecast.
    for split_name, frame in splits.items():
        if len(frame):
            prediction_parts.append(prediction_frame(frame, predict_persistence(frame), 'persistence', 'none', split_name))
    _, ridge_preds = train_ridge(splits['train'], splits['val'], splits['test'], features)
    # Train tabular autoregressive baselines on the shared feature set.
    for split_name, frame in splits.items():
        if len(frame):
            prediction_parts.append(prediction_frame(frame, ridge_preds[split_name], 'ridge_ar', 'none', split_name))
    _, rf_preds = train_random_forest(splits['train'], splits['val'], splits['test'], features, seed=RANDOM_SEED)
    for split_name, frame in splits.items():
        if len(frame):
            prediction_parts.append(prediction_frame(frame, rf_preds[split_name], 'random_forest_ar', 'none', split_name))
    _, xgb_preds = train_xgboost(splits['train'], splits['val'], splits['test'], features, seed=RANDOM_SEED)
    for split_name, frame in splits.items():
        if len(frame):
            prediction_parts.append(prediction_frame(frame, xgb_preds[split_name], 'xgboost_ar', 'none', split_name))
    _, corr_preds = train_correlation_neighbor(splits['train'], splits['val'], splits['test'])
    # Save train-period basin correlations for graph diagnostics and neighbor baselines.
    for split_name, frame in splits.items():
        if len(frame):
            prediction_parts.append(prediction_frame(frame, corr_preds[split_name], 'correlation_neighbor', 'train_positive_corr_lag1', split_name))
    corr = region_correlation_matrix(splits['train'])
    REGION_CORRELATION_MATRIX_CSV.parent.mkdir(parents=True, exist_ok=True)
    # Add a basin-only neural baseline when PyTorch is available.
    corr.to_csv(REGION_CORRELATION_MATRIX_CSV)
    region_correlation_pairs(corr).to_csv(REGION_CORRELATION_PAIRS_CSV, index=False)
    if torch_available() and len(splits['train']):
        _, mlp_preds = train_mlp(splits['train'], splits['val'], splits['test'], features, seed=RANDOM_SEED)
        for split_name, frame in splits.items():
            if len(frame):
                prediction_parts.append(prediction_frame(frame, mlp_preds[split_name], 'basin_only_nn', 'none', split_name))
    else:
    # Collect all baseline predictions in the regional prediction table.
        print('PyTorch is unavailable or train split is empty; skipping basin-only NN.')
    predictions = pd.concat(prediction_parts, ignore_index=True)
    REGION_PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(REGION_PREDICTIONS_CSV, index=False)
    print(f'Saved baseline predictions to {REGION_PREDICTIONS_CSV}')
    display(predictions.head())